# 🧠 Cogito-0.9 API Server — Google Colab Edition
### Free, OpenAI-Compatible REST API

**Runtime**: Use `Runtime → Change runtime type → T4 GPU` for best performance.

This notebook will:
1. Install all dependencies (llama-cpp-python with CUDA)
2. Download Cogito-0.9 GGUF from HuggingFace
3. Start an OpenAI-compatible API server
4. Expose it via Cloudflare/ngrok tunnel
5. Manage API keys with rate limiting

In [ ]:
#@title ⚙️ Configuration
#@markdown Set your preferences. Admin key is auto-generated.

import os, secrets
from pathlib import Path

#@markdown ### Model
QUANT = 'q4_k_m' #@param ["q4_k_m", "q8_0"]
#@markdown - `q4_k_m`: Faster, ~5GB, good quality
#@markdown - `q8_0`: Slower, ~9GB, better quality

#@markdown ### Server
PORT = 8000 #@param {type:"integer"}
MAX_CONTEXT = 4096 #@param {type:"integer"}
MAX_TOKENS = 512 #@param {type:"integer"}
N_GPU_LAYERS = -1 #@param {type:"integer"}
#@markdown Set `N_GPU_LAYERS = -1` to put all layers on GPU

#@markdown ### Tunnel
NGROK_TOKEN = '' #@param {type:"string"}
#@markdown Optional: Get a free token at https://dashboard.ngrok.com

#@markdown ### Rate Limiting
RATE_LIMIT_RPM = 30 #@param {type:"integer"}

# Paths
MODEL_REPO = 'ozaa77/Cogito-0.9'
MODEL_FILE = f'cogito-0.9-{QUANT}.gguf'
MODEL_DIR  = '/content/models'
MODEL_PATH = f'{MODEL_DIR}/{MODEL_FILE}'
API_KEYS_FILE = '/content/api_keys.json'

# Generate admin key
ADMIN_KEY = secrets.token_urlsafe(32)

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

print('=' * 60)
print('⚙️  Cogito-0.9 API — Google Colab Edition')
print('=' * 60)
print(f'  Model:      {MODEL_FILE}')
print(f'  Port:       {PORT}')
print(f'  Context:    {MAX_CONTEXT} tokens')
print(f'  GPU layers: {N_GPU_LAYERS}')
print()
print(f'  🔑 ADMIN KEY: {ADMIN_KEY}')
print(f'  ⚠️  SAVE THIS NOW — it won\'t be shown again!')
print('=' * 60)

In [ ]:
#@title 🖥️ Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:', result.stdout.strip())
else:
    print('⚠️  No GPU detected. Performance will be limited.')
    print('   Go to: Runtime → Change runtime type → T4 GPU')

In [ ]:
#@title 📦 Install Dependencies
import subprocess, sys

print('Installing FastAPI, uvicorn, huggingface_hub...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn[standard]', 'python-multipart',
    'huggingface_hub', 'pydantic', 'requests'
], check=True)
print('✅ Base deps installed')

# Install llama-cpp-python with CUDA
print('\nInstalling llama-cpp-python (CUDA build)...')
print('This may take 2-4 minutes...')

# Try pre-built CUDA wheel first
result = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python',
    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121'
], capture_output=True, text=True)

if result.returncode != 0:
    print('Pre-built wheel failed, building from source...')
    import os
    env = os.environ.copy()
    env['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python',
        '--force-reinstall', '--no-cache-dir'
    ], env=env)

print('✅ llama-cpp-python installed')

In [ ]:
#@title ⬇️ Download Model
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

if Path(MODEL_PATH).exists():
    size_gb = Path(MODEL_PATH).stat().st_size / 1e9
    print(f'✅ Model already downloaded: {size_gb:.2f} GB')
else:
    print(f'⬇️  Downloading {MODEL_FILE}...')
    print('   Tip: Connect Google Drive to persist the model across sessions')
    
    try:
        path = hf_hub_download(
            repo_id=MODEL_REPO,
            filename=MODEL_FILE,
            local_dir=MODEL_DIR,
            local_dir_use_symlinks=False,
        )
        size_gb = Path(path).stat().st_size / 1e9
        print(f'✅ Downloaded: {path} ({size_gb:.2f} GB)')
    except Exception as e:
        print(f'HF Hub failed: {e}, trying wget...')
        url = f'https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_FILE}'
        os.system(f'wget -q --show-progress -O "{MODEL_PATH}" "{url}"')

In [ ]:
#@title 📝 Write Server Files
# The server code is embedded here to avoid external file dependencies

import urllib.request, os

BASE_RAW = 'https://raw.githubusercontent.com/AlGhozaliRamadhan/Cogito/main/server'

for fname in ['api_server.py', 'tunnel_manager.py']:
    dest = f'/content/{fname}'
    try:
        urllib.request.urlretrieve(f'{BASE_RAW}/{fname}', dest)
        print(f'✅ {fname} downloaded')
    except Exception as e:
        print(f'⚠️  Could not auto-download {fname}: {e}')
        print(f'   Upload it manually from the Cogito repository')

In [ ]:
#@title 🚀 Start API Server
import subprocess, time, socket, os

env = os.environ.copy()
env.update({
    'MODEL_PATH': MODEL_PATH,
    'ADMIN_KEY': ADMIN_KEY,
    'API_KEYS_FILE': API_KEYS_FILE,
    'PORT': str(PORT),
    'MAX_CONTEXT': str(MAX_CONTEXT),
    'N_GPU_LAYERS': str(N_GPU_LAYERS),
    'MAX_TOKENS_DEFAULT': str(MAX_TOKENS),
    'RATE_LIMIT_RPM': str(RATE_LIMIT_RPM),
})

server_proc = subprocess.Popen(
    ['python', '/content/api_server.py'],
    env=env,
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)

print('⏳ Waiting for server to start...')
for i in range(60):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            print(f'✅ Server is up on port {PORT}')
            break
    except:
        time.sleep(1)
else:
    print('❌ Server did not start! Check /content/server.log')
    os.system('tail -30 /content/server.log')

In [ ]:
#@title 🌐 Start Public Tunnel
import subprocess, time, json, urllib.request, os, threading
from pathlib import Path

PUBLIC_URL = None

# ── Cloudflared ───────────────────────────────────────────────────────────────
def try_cloudflared():
    global PUBLIC_URL
    cf = '/tmp/cloudflared'
    if not Path(cf).exists():
        urllib.request.urlretrieve(
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            cf
        )
        os.chmod(cf, 0o755)
    
    proc = subprocess.Popen(
        [cf, 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    
    start = time.time()
    while time.time() - start < 30:
        line = proc.stdout.readline()
        if 'trycloudflare.com' in line:
            for part in line.split():
                if part.startswith('https://') and 'trycloudflare' in part:
                    return proc, part.strip()
    return proc, None

# ── ngrok ─────────────────────────────────────────────────────────────────────
def try_ngrok():
    ng = '/tmp/ngrok'
    if not Path(ng).exists():
        tgz = '/tmp/ngrok.tgz'
        urllib.request.urlretrieve(
            'https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz', tgz
        )
        os.system(f'tar -xzf {tgz} -C /tmp/')
    
    if NGROK_TOKEN:
        os.system(f'{ng} config add-authtoken {NGROK_TOKEN}')
    
    proc = subprocess.Popen([ng, 'http', str(PORT)],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    
    for _ in range(10):
        try:
            with urllib.request.urlopen('http://localhost:4040/api/tunnels', timeout=3) as r:
                data = json.loads(r.read())
                for t in data.get('tunnels', []):
                    if t.get('proto') == 'https':
                        return proc, t['public_url']
        except: pass
        time.sleep(1)
    return proc, None

# ── Try each provider ─────────────────────────────────────────────────────────
tunnel_proc = None
for name, fn in [('Cloudflared', try_cloudflared), ('ngrok', try_ngrok)]:
    try:
        print(f'🔄 Trying {name}...')
        proc, url = fn()
        if url:
            PUBLIC_URL = url
            tunnel_proc = proc
            print(f'\n' + '='*60)
            print(f'  ✅  TUNNEL: {name}')
            print(f'  🌐  URL:        {url}')
            print(f'  🔑  ADMIN KEY:  {ADMIN_KEY}')
            print(f'  📖  DOCS:       {url}/docs')
            print(f'  📊  DASHBOARD:  {url}/')
            print('='*60)
            break
        else:
            print(f'⚠️  {name} failed')
    except Exception as e:
        print(f'⚠️  {name} error: {e}')

if not PUBLIC_URL:
    print('❌ No tunnel available. Server runs locally on port', PORT)

In [ ]:
#@title 🔑 Create API Keys
import requests

LOCAL = f'http://localhost:{PORT}'

def create_key(name, role='user', rpm=20):
    r = requests.post(
        f'{LOCAL}/v1/admin/keys/create',
        headers={'Authorization': f'Bearer {ADMIN_KEY}'},
        json={'name': name, 'role': role, 'rate_limit_rpm': rpm}
    )
    return r.json()

# Create some example keys
demo = create_key('demo-user', role='user', rpm=10)
app  = create_key('my-app', role='user', rpm=60)

demo_key = demo['key']['key']
app_key  = app['key']['key']

print('✅ API Keys Created:')
print(f'  [demo-user] {demo_key}')
print(f'  [my-app]    {app_key}')
print()
print('To create more keys, call create_key(name, role, rpm)')

In [ ]:
#@title 🧪 Wait for Model & Test API
import requests, time

LOCAL = f'http://localhost:{PORT}'

print('⏳ Waiting for model to load (this can take 1-3 minutes)...')
for i in range(300):
    try:
        r = requests.get(f'{LOCAL}/health', timeout=3)
        d = r.json()
        if d.get('model_loaded'):
            print(f'\n✅ Model loaded after {i} seconds!')
            break
        print(f'   [{i}s] Loading...', end='\r')
    except: pass
    time.sleep(1)
else:
    print('⚠️  Model taking longer than expected')

# Test
print('\n🧪 Running test inference...')
r = requests.post(
    f'{LOCAL}/v1/chat/completions',
    headers={'Authorization': f'Bearer {demo_key}'},
    json={
        'model': 'cogito-0.9-q4_k_m',
        'messages': [{'role': 'user', 'content': 'Say hi in exactly 5 words.'}],
        'max_tokens': 50,
        'temperature': 0.5,
    }
)

if r.status_code == 200:
    print('🤖 Cogito says:', r.json()['choices'][0]['message']['content'])
    print('\n✅ API is fully working!')
else:
    print(f'❌ Error: {r.status_code} - {r.text}')

In [ ]:
#@title 💓 KeepAlive & Monitor (Run Last)
#@markdown This cell keeps the Colab session alive and monitors server health.
#@markdown Run this last and keep this cell executing.

import time, requests, datetime, threading

LOCAL = f'http://localhost:{PORT}'

print('╔══════════════════════════════════════════════════════╗')
print('║           Cogito-0.9 API — ACTIVE                   ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  URL:       {(PUBLIC_URL or "localhost")[:48].ljust(48)} ║')
print(f'║  Admin Key: {ADMIN_KEY[:48].ljust(48)} ║')
print(f'║  Demo Key:  {demo_key[:48].ljust(48)} ║')
print('╚══════════════════════════════════════════════════════╝')
print()

_ = 0
while True:
    try:
        r = requests.get(f'{LOCAL}/health', timeout=5)
        d = r.json()
        uptime = int(d.get('uptime_seconds', 0))
        h, m, s = uptime//3600, (uptime%3600)//60, uptime%60
        ts = datetime.datetime.now().strftime('%H:%M:%S')
        status = '🟢 Ready' if d['model_loaded'] else '🟡 Loading'
        print(f'[{ts}] {status} | uptime={h}h{m}m{s}s', end='\r')
    except Exception as e:
        print(f'[{datetime.datetime.now().strftime("%H:%M:%S")}] ⚠️  ping failed: {e}', end='\r')
    
    # Busy work to prevent Colab idle
    _ = sum(i**2 for i in range(500_000))
    time.sleep(55)

## 💡 Tips & Tricks

### Persist Model Across Sessions (Google Drive)
```python
from google.colab import drive
drive.mount('/content/drive')
MODEL_DIR = '/content/drive/MyDrive/Cogito'
```

### Extend Session with ngrok
Get a free token at https://dashboard.ngrok.com → paste in `NGROK_TOKEN` above.
ngrok free gives you 1 persistent URL and longer tunnels.

### Use with LangChain
```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    base_url=f"{PUBLIC_URL}/v1",
    api_key=demo_key,
    model="cogito-0.9-q4_k_m",
)
```

### Rate Limiting
Each API key has its own rate limit (requests/minute). Set higher limits for premium users:
```python
create_key('premium-user', rpm=120)
```

### View Server Logs
```python
!tail -50 /content/server.log
```